In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.labeling import rule_intent, INTENTS

df = pd.read_parquet("../data/processed/tickets_sample.parquet")
rules, lf = rule_intent(df["text_clean"])
df = df.join(rules)

covered = df["rule_label"].notna()
print("Coverage (≥1 rule fired):", round(covered.mean(), 3))
print("\nRule label distribution:\n", df["rule_label"].value_counts(dropna=False))
print("\nConflict rate among covered:", round((df.loc[covered, "rule_agreement"] < 1).mean(), 3))

lf_cov = lf.mean().sort_values(ascending=False).round(3)
lf_cov.to_csv("../reports/intent_lf_coverage.csv")
print("\nTop 5 rules:\n", lf_cov.head(5))
print("\nWeakest 5 rules:\n", lf_cov.tail(5))

for lab in INTENTS:
    print(f"\n--- {lab} ---")
    for t in df.loc[df["rule_label"] == lab, "text_clean"].sample(3, random_state=1):
        print(" •", t)

Coverage (≥1 rule fired): 0.576

Rule label distribution:
 rule_label
NaN          42423
General      21862
Technical    11615
Billing      10196
Complaint     8567
Account       5337
Name: count, dtype: int64

Conflict rate among covered: 0.086

Top 5 rules:
 General_1      0.179
General_2      0.101
Technical_4    0.054
General_3      0.050
Complaint_1    0.048
dtype: float64

Weakest 5 rules:
 Account_4      0.008
Account_2      0.008
Account_5      0.005
Complaint_2    0.005
Technical_5    0.004
dtype: float64

--- Billing ---
 • where does the from ticket price hikes go for an 8 month pregnant woman to stand up squashed against the toilet for 3hrs
 • so let me get this straight you charge me 25 to get my money in 30 minutes then say it will be 24 hours stop stealing
 • morning just to advise ticket machine at how wood not issuing tickets after payment will payment go through thanks

--- Account ---
 • has my account met the requirements for the 10 000 bonus star points please rese

In [2]:
import time
from transformers import pipeline
from src.cleaning import URL_RE, MENTION_RE

# Descriptive phrases work better for zero-shot than bare class names
ZS_LABELS = {
    "a billing, payment or refund issue": "Billing",
    "a technical problem with a product, app or service": "Technical",
    "an account, login or password issue": "Account",
    "a complaint about bad service or experience": "Complaint",
    "a general question or inquiry": "General",
}
HYPOTHESIS = "This customer message is about {}."

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="mps")

# NLI models read natural text best: raw tweet minus URLs/@mentions
df["text_zs"] = (df["text"].str.replace(URL_RE, " ", regex=True)
                           .str.replace(MENTION_RE, " ", regex=True).str.strip())

test = df.head(512)
t0 = time.time()
out = zs(test["text_zs"].tolist(), candidate_labels=list(ZS_LABELS),
         hypothesis_template=HYPOTHESIS, batch_size=32)
elapsed = time.time() - t0
print(f"{elapsed:.1f}s for 512 → est. {elapsed / 512 * 100_000 / 60:.0f} min for 100k")

chk = test.assign(zs_label=[ZS_LABELS[o["labels"][0]] for o in out],
                  zs_conf=[round(o["scores"][0], 3) for o in out])
fired = chk["rule_label"].notna()
print("\nAgreement where rules fired:",
      round((chk.loc[fired, "rule_label"] == chk.loc[fired, "zs_label"]).mean(), 3))
print("\nZero-shot distribution:\n", chk["zs_label"].value_counts())
print("\nZero-shot confidence:\n", chk["zs_conf"].describe().round(2))

/Users/naren/Projects/Intelligent_Customer_Support_Ticket_System_Final_Project/Intelligent_Customer_Support_Ticket_System_Final_Project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1101 17:11:28.115000 6486 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 10337.23it/s]


54.6s for 512 → est. 178 min for 100k

Agreement where rules fired: 0.529

Zero-shot distribution:
 zs_label
General      236
Complaint    195
Technical     63
Billing        9
Account        9
Name: count, dtype: int64

Zero-shot confidence:
 count    512.00
mean       0.61
std        0.18
min        0.28
25%        0.46
50%        0.57
75%        0.76
max        0.98
Name: zs_conf, dtype: float64


In [4]:
%pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
ZS_LABELS_V2 = {
    "being charged, billing, payment or a refund": "Billing",
    "a technical fault, error or something not working": "Technical",
    "logging in, a password or managing an account": "Account",
    "anger or dissatisfaction with the service": "Complaint",
    "asking a general question or giving feedback": "General",
}

zs2 = pipeline("zero-shot-classification",
               model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0", device="mps")

def run_zs(pipe, labels, texts, multi):
    t0 = time.time()
    out = pipe(texts, candidate_labels=list(labels), hypothesis_template=HYPOTHESIS,
               multi_label=multi, batch_size=32)
    return [labels[o["labels"][0]] for o in out], time.time() - t0

texts = test["text_zs"].tolist()
for name, pipe, labels, multi in [
    ("BART + new labels   ", zs,  ZS_LABELS_V2, False),
    ("DeBERTa + new labels", zs2, ZS_LABELS_V2, True),
]:
    pred, sec = run_zs(pipe, labels, texts, multi)
    p = pd.Series(pred, index=test.index)
    agree = (test.loc[fired, "rule_label"] == p[fired]).mean()
    print(f"{name}: {sec:.0f}s ({sec / 512 * 100_000 / 60:.0f} min/100k) | "
          f"agreement {agree:.3f} | {p.value_counts().to_dict()}")

/Users/naren/Projects/Intelligent_Customer_Support_Ticket_System_Final_Project/Intelligent_Customer_Support_Ticket_System_Final_Project/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 18695.24it/s]


BART + new labels   : 48s (157 min/100k) | agreement 0.507 | {'Complaint': 252, 'General': 140, 'Technical': 106, 'Billing': 9, 'Account': 5}
DeBERTa + new labels: 15s (50 min/100k) | agreement 0.591 | {'Complaint': 166, 'Technical': 144, 'General': 138, 'Billing': 52, 'Account': 12}


In [6]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

CHUNK = 5000
ZS_DIR = Path("../data/processed/zs_chunks")
ZS_DIR.mkdir(exist_ok=True)

for start in range(0, len(df), CHUNK):
    f = ZS_DIR / f"zs_{start:06d}.parquet"
    if f.exists():                      # resume: skip finished chunks
        continue
    part = df.iloc[start:start + CHUNK]
    out = zs2(part["text_zs"].tolist(), candidate_labels=list(ZS_LABELS_V2),
              hypothesis_template=HYPOTHESIS, multi_label=True, batch_size=32)
    pd.DataFrame({
        "tweet_id":  part["tweet_id"].values,
        "zs_label":  [ZS_LABELS_V2[o["labels"][0]] for o in out],
        "zs_conf":   [o["scores"][0] for o in out],
        "zs_margin": [o["scores"][0] - o["scores"][1] for o in out],  # gap to runner-up
    }).to_parquet(f, index=False)
    print(f"{start + len(part):,}/{len(df):,} done", flush=True)

5,000/100,000 done
10,000/100,000 done
15,000/100,000 done
20,000/100,000 done
25,000/100,000 done
30,000/100,000 done
35,000/100,000 done
40,000/100,000 done
45,000/100,000 done
50,000/100,000 done
55,000/100,000 done
60,000/100,000 done
65,000/100,000 done
70,000/100,000 done
75,000/100,000 done
80,000/100,000 done
85,000/100,000 done
90,000/100,000 done
95,000/100,000 done
100,000/100,000 done


In [7]:
zs_df = pd.concat(pd.read_parquet(f) for f in sorted(ZS_DIR.glob("zs_*.parquet")))
df = df.drop(columns=["zs_label", "zs_conf", "zs_margin"], errors="ignore").merge(zs_df, on="tweet_id", how="left")
assert len(df) == 100_000 and df["zs_label"].notna().all()

fired = df["rule_label"].notna()
agree = fired & (df["rule_label"] == df["zs_label"])
print(f"Rules fired: {fired.sum():,} | agree with zero-shot: {agree.sum():,} ({agree.sum() / fired.sum():.1%})")

print("\nAgreement rate by rule label:\n",
      agree[fired].groupby(df.loc[fired, "rule_label"]).mean().round(3))

print("\nTop disagreements (rule → zero-shot):\n",
      df[fired & ~agree].groupby(["rule_label", "zs_label"]).size()
        .sort_values(ascending=False).head(8))

print("\nZero-shot confidence on rows NO rule covered:\n",
      df.loc[~fired, ["zs_conf", "zs_margin"]].describe().round(2))

for c, m in [(0.7, 0.2), (0.8, 0.3), (0.9, 0.4)]:
    keep = ~fired & (df["zs_conf"] >= c) & (df["zs_margin"] >= m)
    print(f"conf≥{c}, margin≥{m}: {keep.sum():>6,} uncovered rows | {df.loc[keep, 'zs_label'].value_counts().to_dict()}")
    

Rules fired: 57,577 | agree with zero-shot: 32,294 (56.1%)

Agreement rate by rule label:
 rule_label
Account      0.299
Billing      0.480
Complaint    0.799
General      0.512
Technical    0.669
dtype: float64

Top disagreements (rule → zero-shot):
 rule_label  zs_label 
General     Complaint    5378
            Technical    3760
Billing     Complaint    3331
Technical   Complaint    2485
Account     Technical    1498
General     Billing      1465
Account     Complaint    1254
Technical   General      1227
dtype: int64

Zero-shot confidence on rows NO rule covered:
         zs_conf  zs_margin
count  42423.00   42423.00
mean       0.48       0.39
std        0.45       0.42
min        0.00       0.00
25%        0.01       0.01
50%        0.38       0.14
75%        0.98       0.90
max        1.00       1.00
conf≥0.7, margin≥0.2: 16,568 uncovered rows | {'Complaint': 9692, 'Technical': 4000, 'General': 2380, 'Billing': 447, 'Account': 49}
conf≥0.8, margin≥0.3: 15,324 uncovered rows | {'C

In [8]:
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(exist_ok=True)

# Gold: random 400 from the full sample (evaluation only)
gold = df.sample(n=400, random_state=7)

# Silver: rows where rules and zero-shot agree, minus gold rows
silver = df[agree & ~df["tweet_id"].isin(gold["tweet_id"])].copy()
silver["intent"] = silver["rule_label"]
print("Silver intent rows:", len(silver))
print(silver["intent"].value_counts(normalize=True).round(3))

df.to_parquet("../data/processed/tickets_weak_labels.parquet", index=False)
silver.to_parquet("../data/processed/silver_intent.parquet", index=False)

# Labeling sheet: text only, so you're not anchored by the rule/zero-shot labels
(gold[["tweet_id", "text"]]
 .assign(intent="", priority="")
 .to_csv(GOLD_DIR / "gold_to_label.csv", index=False))
print("Gold sheet saved:", GOLD_DIR / "gold_to_label.csv")

Silver intent rows: 32168
intent
General      0.347
Technical    0.240
Complaint    0.212
Billing      0.152
Account      0.049
Name: proportion, dtype: float64
Gold sheet saved: ../data/gold/gold_to_label.csv


In [3]:
from src.labeling import rule_priority

df = pd.read_parquet("../data/processed/tickets_weak_labels.parquet")
pr = rule_priority(df["text_zs"], df["zs_label"])
df = df.drop(columns=pr.columns, errors="ignore").join(pr)

print(df["rule_priority"].value_counts(normalize=True).round(3))
for lab in ["High", "Medium", "Low"]:
    print(f"\n--- {lab} ---")
    for t in df.loc[df["rule_priority"] == lab, "text_zs"].sample(4, random_state=3):
        print(" •", t[:160])

rule_priority
Medium    0.562
Low       0.374
High      0.064
Name: proportion, dtype: float64

--- High ---
 • It is outrageous that   think it is acceptable to cancel my (nearly expired) debit card &amp; make me wait three days for the quickest delivery of a new one, be
 • I'm heartbroken! All my voicemails from my aunt who passed away have been deleted! I had them locked, this shouldn't have happened!
 • No thanks. Just got off the phone with y’all. Automated notification that there is an outage and y’all have no further info. 🙄
 • Beware #Target online. Order accidentally to wrong address. #Target refuses to cancel. Ask for supervisor, hung up on. #TargetSucks

--- Medium ---
 • still having a problem getting a black screen after every game while in a party....
 • we are due into Euston at 00:03 and missed last train to ebsfleet if we arrange are own cab to not queue can we claim it back?
 • i've placed a same day order &amp; despite me changing the address TWICE, it is showing as 

In [4]:
import time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from transformers import pipeline

df.to_parquet("../data/processed/tickets_weak_labels.parquet", index=False)  # save priority signals

zs2 = pipeline("zero-shot-classification",
               model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0", device="mps")
PRIO_LABELS = {
    "an urgent problem that needs immediate attention": "High",
    "a problem that needs fixing but is not urgent": "Medium",
    "a simple question, feedback or thanks with no urgency": "Low",
}
PRIO_HYP = "This customer message describes {}."

def zs_priority(texts):
    out = zs2(texts, candidate_labels=list(PRIO_LABELS), hypothesis_template=PRIO_HYP,
              multi_label=True, batch_size=32)
    return pd.DataFrame({"zs_priority": [PRIO_LABELS[o["labels"][0]] for o in out],
                         "zs_p_conf":   [o["scores"][0] for o in out],
                         "zs_p_margin": [o["scores"][0] - o["scores"][1] for o in out]})

t0 = time.time()
chk = zs_priority(df["text_zs"].head(512).tolist())
chk["rule_priority"] = df["rule_priority"].head(512).values
print(f"{time.time() - t0:.0f}s for 512 → est. {(time.time() - t0) / 512 * 100_000 / 60:.0f} min for 100k")
print("Agreement with rules:", round((chk["zs_priority"] == chk["rule_priority"]).mean(), 3))
print(pd.crosstab(chk["rule_priority"], chk["zs_priority"], margins=True))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 10710.44it/s]


7s for 512 → est. 24 min for 100k
Agreement with rules: 0.568
zs_priority    High  Low  Medium  All
rule_priority                        
High              8    2      11   21
Low              12  143      46  201
Medium           65   85     140  290
All              85  230     197  512


In [5]:
PR_DIR = Path("../data/processed/zs_priority_chunks")
PR_DIR.mkdir(exist_ok=True)
for start in range(0, len(df), 5000):
    f = PR_DIR / f"zsp_{start:06d}.parquet"
    if f.exists():
        continue
    part = df.iloc[start:start + 5000]
    zs_priority(part["text_zs"].tolist()).assign(tweet_id=part["tweet_id"].values).to_parquet(f, index=False)
    print(f"{start + len(part):,}/{len(df):,} done", flush=True)

5,000/100,000 done
10,000/100,000 done
15,000/100,000 done
20,000/100,000 done
25,000/100,000 done
30,000/100,000 done
35,000/100,000 done
40,000/100,000 done
45,000/100,000 done
50,000/100,000 done
55,000/100,000 done
60,000/100,000 done
65,000/100,000 done
70,000/100,000 done
75,000/100,000 done
80,000/100,000 done
85,000/100,000 done
90,000/100,000 done
95,000/100,000 done
100,000/100,000 done


In [6]:
import numpy as np

pr_df = pd.concat(pd.read_parquet(f) for f in sorted(PR_DIR.glob("zsp_*.parquet")))
df = (df.drop(columns=["zs_priority", "zs_p_conf", "zs_p_margin"], errors="ignore")
        .merge(pr_df, on="tweet_id", how="left"))
assert len(df) == 100_000 and df["zs_priority"].notna().all()
df.to_parquet("../data/processed/tickets_weak_labels.parquet", index=False)

agree_p = df["rule_priority"] == df["zs_priority"]
print(f"Agreement: {agree_p.mean():.1%}\n")
print(pd.crosstab(df["rule_priority"], df["zs_priority"], margins=True), "\n")

urgency = ((df["vader"] <= -0.6).astype(int) + df["escalation_kw"].astype(int)
           + (df["n_exclaim"] >= 2).astype(int) + (df["caps_ratio"] >= 0.3).astype(int))
extra_high = (~agree_p & (df["zs_priority"] == "High") & (df["rule_priority"] == "Medium")
              & (df["zs_p_conf"] >= 0.9) & (df["zs_p_margin"] >= 0.3) & (urgency >= 1))
print(f"Agreed High: {(agree_p & (df['zs_priority'] == 'High')).sum():,} | "
      f"exception High: {extra_high.sum():,}")
print("\nSample exception-High rows:")
for t in df.loc[extra_high, "text_zs"].sample(6, random_state=5):
    print(" •", t[:160])

# Silver priority set (gold excluded)
gold_ids = set(pd.read_csv("../data/gold/gold_to_label.csv", usecols=["tweet_id"])["tweet_id"])
keep = (agree_p | extra_high) & ~df["tweet_id"].isin(gold_ids)
silver_p = df[keep].copy()
silver_p["priority"] = np.where(extra_high[keep], "High", silver_p["rule_priority"])
silver_p["priority_source"] = np.where(extra_high[keep], "zs_high_exception", "agree")

print(f"\nSilver priority rows: {len(silver_p):,}")
print(silver_p["priority"].value_counts(normalize=True).round(3))
silver_p.to_parquet("../data/processed/silver_priority.parquet", index=False)

Agreement: 55.4%

zs_priority     High    Low  Medium     All
rule_priority                              
High            2487   1221    2736    6444
Low             2875  26327    8157   37359
Medium         13825  15777   26595   56197
All            19187  43325   37488  100000 

Agreed High: 2,487 | exception High: 304

Sample exception-High rows:
 • So...my son wakes me up in the middle of the night because his IPOD is on fire! And his room smells like fumes!!
 • #apple #ios11   Please, hurry up! iOS 11 is bullying my iDevices. Ridiculous battery life and super glitches... come on!
 • jai besoin d'aide urgent merci
 • In bed sick. Checked tracking on item 2 arrive by 9p. See pic. 2 mean dogs. Not a sound 3rd problem with AMZL. Now they lie 2! 1
 • please save my life and hold my 4:25 plane from Midway to Detroit!! Checking in now but crazy traffic getting here!
 • For the past 2-3 days i have been experiencing problems with my Idea Network with connection to internet and calling s